# Data, integration and exploratory analysis

Run after group-owned collection and preparation. Every output must be interpreted and checked by the group. No results are supplied. The rectangle is around Switzerland, not a national boundary; the nearest selected airport is not an inferred origin/destination.

In [ ]:
from contextlib import closing
from pathlib import Path
import sqlite3
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
ROOT = Path.cwd() if (Path.cwd() / "config.json").exists() else Path.cwd().parent
DB = ROOT / "data/processed/aircraft.sqlite"
if not DB.exists():
    raise RuntimeError("Collect real data and run src/prepare.py first.")
with closing(sqlite3.connect(DB)) as db:
    data = pd.read_sql_query("SELECT * FROM observations", db)
    airports = pd.read_sql_query("SELECT * FROM airports", db)
    audit = pd.read_sql_query("SELECT * FROM integration_audit", db)
data["observed_at_utc"] = pd.to_datetime(data.observed_at_utc, utc=True)
OUT = ROOT / "outputs"
(OUT / "figures").mkdir(parents=True, exist_ok=True)
(OUT / "tables").mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


In [ ]:
from contextlib import closing
display(audit)
display(data.isna().sum().to_frame("missing_after_preparation"))
with closing(sqlite3.connect(DB)) as db:
    coverage = pd.read_sql_query("""SELECT date_utc, COUNT(*) observations,
        COUNT(DISTINCT icao24) aircraft FROM observations GROUP BY date_utc ORDER BY date_utc""", db)
    summary = pd.read_sql_query("""SELECT a.name, o.zone, COUNT(*) observations,
        COUNT(DISTINCT o.icao24) aircraft, AVG(o.velocity) mean_speed_m_s
        FROM observations o JOIN airports a ON o.airport_ident=a.ident
        GROUP BY a.name, o.zone""", db)
display(coverage); display(summary)
coverage.to_csv(OUT / "tables/date_coverage.csv", index=False)
summary.to_csv(OUT / "tables/airport_zone_summary.csv", index=False)
display(data.groupby("zone")[["velocity", "geo_altitude", "distance_km"]].describe())
# Optional third source: the wind join audit is part of data quality (docs/WEATHER.md).
with closing(sqlite3.connect(DB)) as db:
    if db.execute("SELECT COUNT(*) FROM sqlite_master WHERE name='weather_integration_audit'").fetchone()[0]:
        wind_audit = pd.read_sql_query("SELECT * FROM weather_integration_audit", db)
        display(wind_audit)
        wind_audit.to_csv(OUT / "tables/weather_integration_audit.csv", index=False)
    else:
        print("No wind audit. Optional: run src/weather.py fetch, then src/prepare.py.")


In [ ]:
# Acquisition coverage includes failures and empty snapshots, not just retained rows.
manifest = pd.read_json(ROOT / "data/raw/manifest.jsonl", lines=True)
requests_log = manifest[manifest.source_url.str.contains("states/all", regex=False)].copy()
requests_log["retrieved_at_utc"] = pd.to_datetime(requests_log.retrieved_at_utc, utc=True)
# HTTP 200 alone is insufficient: schema validation must succeed and raw bytes must be saved.
requests_log["successful"] = requests_log.status.eq(200) & requests_log.path.notna()
requests_log["attempted"] = requests_log.get("outcome", pd.Series(index=requests_log.index, dtype=str)).ne("deferred")
print("Invocations:", len(requests_log), "Attempted:", requests_log.attempted.sum(), "Successful:", requests_log.successful.sum())
requests_log.set_index("retrieved_at_utc")[["attempted", "successful"]].resample("h").sum().plot(figsize=(12,4))
plt.ylabel("Collection attempts / successes"); plt.tight_layout()
plt.savefig(OUT / "figures/collection_coverage.png", dpi=160)
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
sns.histplot(data=data, x="velocity", hue="zone", element="step", stat="density", common_norm=False, ax=ax[0])
sns.histplot(data=data, x="geo_altitude", hue="zone", element="step", stat="density", common_norm=False, ax=ax[1])
ax[0].set_xlabel("Ground speed (m/s)"); ax[1].set_xlabel("Geometric altitude (m)")
fig.tight_layout(); fig.savefig(OUT / "figures/group_distributions.png", dpi=160); plt.show()
plot_sample = data.sample(min(5000,len(data)), random_state=42)
sns.scatterplot(data=plot_sample, x="geo_altitude", y="velocity", hue="zone", alpha=0.25, s=12)
plt.xlabel("Geometric altitude (m)"); plt.ylabel("Ground speed (m/s)")
plt.tight_layout(); plt.savefig(OUT / "figures/altitude_speed.png", dpi=160); plt.show()
# Check overlapping altitude support before comparing speed.
data["altitude_band_m"] = pd.cut(data.geo_altitude, bins=np.arange(-1000,21001,2000))
banded = data.groupby(["zone", "altitude_band_m"], observed=True).agg(
    observations=("icao24","size"), aircraft=("icao24","nunique"), median_speed_m_s=("velocity","median"))
display(banded)
banded.to_csv(OUT / "tables/altitude_band_comparison.csv")

In [ ]:
import folium
config = json.loads((ROOT / "config.json").read_text())
colors = {"near":"#d97706", "middle":"#64748b", "far":"#2563eb"}
m = folium.Map(location=[46.8,8.2], zoom_start=7)
for airport in airports.itertuples():
    folium.Marker([airport.latitude_deg,airport.longitude_deg], tooltip=airport.ident).add_to(m)
for row in data.sample(min(1500,len(data)), random_state=42).itertuples():
    folium.CircleMarker([row.latitude,row.longitude],radius=2,color=colors[row.zone],
        fill=True,fill_opacity=.5,tooltip=f"{row.zone}; {row.velocity:.1f} m/s; nearest selected airport {row.airport_ident}").add_to(m)
# Explicit legend so colors are interpretable outside the notebook.
from branca.element import Element
m.get_root().html.add_child(Element('<div style="position:fixed;bottom:30px;left:30px;background:white;padding:12px;z-index:9999">'
    f'<b>Proximity to selected airports</b><br><span style="color:#d97706">Near: <= {config["near_km"]} km</span><br>'
    f'<span style="color:#64748b">Middle: {config["near_km"]}-{config["far_km"]} km</span><br><span style="color:#2563eb">Far: >= {config["far_km"]} km</span><br>'
    'Sample of observations; ground speed in m/s</div>'))
m.save(str(OUT / "figures/aircraft_map.html"))
m

## Group interpretation required

Describe coverage, exclusions, counts of distinct aircraft, sample imbalance and common altitude support. Explain map sampling, repeated observations and ADS-B coverage bias. Check map legend thresholds if config changes. Add an aircraft-balanced sensitivity comparison before final conclusions. From the wind audit, report the match fraction, the interpolated share and the share of level gaps over 500 m; unmatched or unbracketed rows weaken every wind statement below.

## Optional aircraft metadata

A separate aircraft table is left-joined by normalized ICAO24. Unmatched reports stay in the analysis. Key matches do not prove current or complete metadata: the downloaded reference is old and mixes sources. Compare observation and distinct-aircraft coverage, inspect the unknown/other category, and distinguish observation frequency from aircraft frequency.

In [ ]:
from contextlib import closing
with closing(sqlite3.connect(DB)) as db:
    has_metadata = db.execute("SELECT COUNT(*) FROM sqlite_master WHERE name='aircraft_metadata'").fetchone()[0]
    if has_metadata:
        metadata_audit = pd.read_sql_query("SELECT * FROM metadata_integration_audit", db)
        types = pd.read_sql_query("""SELECT COALESCE(NULLIF(m.aircraft_typecode,''),'unknown/other') typecode,
            COALESCE(m.aircraft_category,'unknown/other') aircraft_category,
            COUNT(*) observations, COUNT(DISTINCT o.icao24) aircraft
            FROM observations o LEFT JOIN aircraft_metadata m ON o.icao24=m.icao24
            GROUP BY 1,2 ORDER BY aircraft DESC""", db)
        display(metadata_audit); display(types.head(20))
        types.to_csv(OUT / "tables/aircraft_type_coverage.csv", index=False)
    else:
        print("Optional metadata tables absent. Existing EDA is still valid; rebuild with src/prepare.py for enrichment.")


## Wind and aircraft type

Ground speed = airspeed + wind along the track. This section describes how much of the ground-speed pattern is modelled wind and how aircraft categories are spread over the zones. Everything here is **descriptive**.

- Wind is model output (Open-Meteo, hourly, 0.5° grid, interpolated between pressure levels; see `docs/WEATHER.md`), not the wind the aircraft experienced.
- `tailwind_ms`: + = wind pushes along the track, − = headwind. `airspeed_est_ms` = ground velocity minus wind vector. It is computed **from the target** and must never be a model feature.
- The same aircraft appears in many snapshots, so observations are not independent; aircraft counts are shown next to observation counts.
- Only rows with `wind_matched = 1` enter the wind blocks. Aircraft metadata covers only part of the fleet; `unknown/other` stays a separate group.

In [ ]:
from contextlib import closing
# Block 1: wind by altitude band and zone (SQL aggregation on the wind view).
ZONES = ["near", "middle", "far"]
BAND_M = 2000  # same band edges as the altitude-band table above (-1000, 1000, 3000, ... m); left-closed here
BAND_SQL = f"-1000 + {BAND_M} * CAST((geo_altitude + 1000) / {BAND_M} AS INTEGER)"
with closing(sqlite3.connect(DB)) as db:
    views = set(pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='view'", db).name)
    matched = (db.execute("SELECT COALESCE(SUM(wind_matched), 0) FROM observations_with_wind").fetchone()[0]
               if "observations_with_wind" in views else 0)
    HAS_WIND = matched > 0
    if HAS_WIND:
        wind_by_band_zone = pd.read_sql_query(f"""
            SELECT {BAND_SQL} AS band_low_m, {BAND_SQL} + {BAND_M} AS band_high_m, zone,
                   COUNT(*) AS observations, COUNT(DISTINCT icao24) AS aircraft,
                   AVG(wind_speed_ms) AS mean_wind_speed_ms, AVG(tailwind_ms) AS mean_tailwind_ms,
                   AVG(CASE WHEN tailwind_ms IS NULL THEN NULL WHEN tailwind_ms < 0 THEN 1.0 ELSE 0.0 END)
                       AS headwind_share,
                   AVG(wind_interpolated) AS interpolated_share
            FROM observations_with_wind WHERE wind_matched = 1
            GROUP BY 1, 2, 3""", db)
        # Interpolation does not depend on the zone, so its quality is also shown per altitude band alone.
        interpolation_by_band = pd.read_sql_query(f"""
            SELECT {BAND_SQL} AS band_low_m, {BAND_SQL} + {BAND_M} AS band_high_m,
                   COUNT(*) AS observations, AVG(wind_interpolated) AS interpolated_share,
                   MAX(wind_level_gap_m) AS max_level_gap_m
            FROM observations_with_wind WHERE wind_matched = 1
            GROUP BY 1, 2 ORDER BY 1""", db)
        wind_obs = pd.read_sql_query(f"""
            SELECT icao24, time_position, zone, latitude, longitude, geo_altitude, velocity, true_track,
                   {BAND_SQL} AS band_low_m, wind_time_utc, wind_speed_ms, wind_dir_deg,
                   tailwind_ms, crosswind_ms, airspeed_est_ms, wind_interpolated
            FROM observations_with_wind WHERE wind_matched = 1""", db)
if HAS_WIND:
    print(f"Observations with matched wind: {matched} of {len(data)}")
    wind_by_band_zone["zone"] = pd.Categorical(wind_by_band_zone.zone, ZONES, ordered=True)
    wind_by_band_zone = wind_by_band_zone.sort_values(["band_low_m", "zone"]).reset_index(drop=True)
    display(wind_by_band_zone.round(3)); display(interpolation_by_band.round(3))
    wind_by_band_zone.to_csv(OUT / "tables/wind_by_altitude_zone.csv", index=False)
    interpolation_by_band.to_csv(OUT / "tables/wind_interpolation_by_altitude.csv", index=False)
else:
    print("No matched wind rows. Optional: run src/weather.py fetch, then src/prepare.py. Wind blocks are skipped.")


### Interpretation (group): wind by altitude band and zone

*Placeholder.* How strong is the wind per altitude band, and how much of it acts along the track (mean tailwind, headwind share)? Do near and far differ within the same band, or only because their altitude mix differs? Which bands have too few aircraft to say anything? Is the interpolated share high in every band with data?

In [ ]:
# Block 2: ground speed vs estimated airspeed over altitude (descriptive only, airspeed_est_ms is never a feature).
if HAS_WIND:
    MIN_BAND_OBS = 5  # medians from fewer observations are not drawn
    speeds = wind_obs.melt(id_vars=["band_low_m", "geo_altitude"], value_vars=["velocity", "airspeed_est_ms"],
                           var_name="measure", value_name="speed_ms").dropna(subset=["speed_ms"])
    labels = {"velocity": "Ground speed (observed)", "airspeed_est_ms": "Estimated airspeed (ground speed minus wind)"}
    band_medians = speeds.groupby(["measure", "band_low_m"]).speed_ms.agg(median="median", observations="size").reset_index()
    band_medians = band_medians[band_medians.observations >= MIN_BAND_OBS]
    fig, ax = plt.subplots(figsize=(10, 5))
    for (measure, label), color in zip(labels.items(), ["#2563eb", "#d97706"]):
        points = speeds[speeds.measure == measure]
        ax.scatter(points.geo_altitude, points.speed_ms, s=8, alpha=0.15, color=color)
        medians = band_medians[band_medians.measure == measure]
        ax.plot(medians.band_low_m + BAND_M / 2, medians["median"], marker="o", color=color,
                label=f"{label}: median per {BAND_M} m band")
    ax.set_xlabel("Geometric altitude (m)"); ax.set_ylabel("Speed (m/s)")
    ax.set_title("Ground speed vs. estimated airspeed by altitude (descriptive, model wind)")
    ax.legend(title=f"Points = single observations; bands with < {MIN_BAND_OBS} obs. not drawn", loc="lower right")
    fig.tight_layout(); fig.savefig(OUT / "figures/ground_vs_airspeed_altitude.png", dpi=160); plt.show()
    display(band_medians.pivot(index="band_low_m", columns="measure", values="median").round(1))


### Interpretation (group): ground speed vs estimated airspeed

*Placeholder.* At which altitudes do the two curves separate, and in which direction? Is the gap consistent with the wind table above? Keep it descriptive: the estimate inherits all model-wind errors, and it must not be used as a feature or as a target substitute.

In [ ]:
# Block 3: tailwind by flight direction. Sectors of true_track: East 45-135°, West 225-315°, rest North/South.
DIRECTIONS = ["East", "West", "North/South"]


def flight_direction(track_deg):
    """Direction sector of the true track (degrees clockwise from north); missing track stays missing."""
    track = pd.Series(track_deg, dtype=float) % 360
    sector = np.select([track.between(45, 135), track.between(225, 315)], ["East", "West"], "North/South")
    return pd.Series(sector, index=track.index).where(track.notna())


if HAS_WIND:
    wind_obs["direction"] = pd.Categorical(flight_direction(wind_obs.true_track), DIRECTIONS)
    directed = wind_obs.dropna(subset=["direction", "tailwind_ms"]).copy()
    print("Excluded (missing true_track or tailwind):", len(wind_obs) - len(directed))
    directed["altitude_band"] = (directed.band_low_m.astype(int).astype(str) + "–"
                                 + (directed.band_low_m + BAND_M).astype(int).astype(str) + " m")
    band_order = sorted(directed.altitude_band.unique(), key=lambda s: int(s.split("–")[0]))
    tailwind_by_direction = directed.groupby(["direction", "band_low_m"], observed=True).agg(
        observations=("icao24", "size"), aircraft=("icao24", "nunique"),
        median_tailwind_ms=("tailwind_ms", "median"), median_crosswind_ms=("crosswind_ms", "median")).reset_index()
    display(tailwind_by_direction.round(1))
    tailwind_by_direction.to_csv(OUT / "tables/tailwind_by_direction.csv", index=False)
    palette = {"East": "#2563eb", "West": "#d97706", "North/South": "#64748b"}
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    sns.histplot(data=directed, x="tailwind_ms", hue="direction", hue_order=DIRECTIONS, palette=palette,
                 element="step", stat="density", common_norm=False, ax=ax[0])
    ax[0].axvline(0, color="black", lw=0.8)
    ax[0].get_legend().set_title("Direction (true track)")
    ax[0].set_xlabel("Tailwind along track (m/s; + = tailwind, − = headwind)"); ax[0].set_ylabel("Density")
    ax[0].set_title("Tailwind distribution by flight direction")
    sns.boxplot(data=directed, x="altitude_band", y="tailwind_ms", hue="direction", hue_order=DIRECTIONS,
                order=band_order, palette=palette, ax=ax[1])
    ax[1].axhline(0, color="black", lw=0.8)
    ax[1].set_xlabel("Geometric altitude band"); ax[1].set_ylabel("Tailwind along track (m/s)")
    ax[1].set_title("Tailwind by altitude band and flight direction"); ax[1].tick_params(axis="x", rotation=30)
    ax[1].legend(title="Direction (true track)")
    fig.tight_layout(); fig.savefig(OUT / "figures/tailwind_by_direction.png", dpi=160); plt.show()


### Interpretation (group): tailwind by flight direction

*Placeholder.* With a westerly jet stream, eastbound flights should gain and westbound flights lose ground speed; does the data show this, and from which altitude on? North/South flights mostly feel crosswind (see `median_crosswind_ms`). How many aircraft (not observations) are behind each box?

In [ ]:
# Block 4: share of observations and of distinct aircraft per category and zone (SQL with window functions).
from contextlib import closing
with closing(sqlite3.connect(DB)) as db:
    has_enriched = db.execute("SELECT COUNT(*) FROM sqlite_master WHERE name='observations_enriched'").fetchone()[0]
    if has_enriched:
        category_zone = pd.read_sql_query("""
            SELECT zone, COALESCE(NULLIF(aircraft_category, ''), 'unknown/other') AS aircraft_category,
                   COUNT(*) AS observations, COUNT(DISTINCT icao24) AS aircraft,
                   1.0 * COUNT(*) / SUM(COUNT(*)) OVER (PARTITION BY zone) AS observation_share,
                   1.0 * COUNT(DISTINCT icao24) / SUM(COUNT(DISTINCT icao24)) OVER (PARTITION BY zone) AS aircraft_share
            FROM observations_enriched GROUP BY 1, 2""", db)
if has_enriched:
    # An aircraft seen in two zones is counted in both; aircraft shares are within each zone.
    category_zone["zone"] = pd.Categorical(category_zone.zone, ZONES, ordered=True)
    category_zone = category_zone.sort_values(["zone", "observations"], ascending=[True, False]).reset_index(drop=True)
    display(category_zone.round(3))
    category_zone.to_csv(OUT / "tables/category_share_by_zone.csv", index=False)
    categories = sorted(category_zone.aircraft_category.unique(), key=lambda c: (c == "unknown/other", c))
    fig, ax = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
    for axis, column, title in [(ax[0], "observation_share", "Share of observations"),
                                (ax[1], "aircraft_share", "Share of distinct aircraft")]:
        shares = category_zone.pivot(index="zone", columns="aircraft_category", values=column).reindex(
            index=ZONES, columns=categories).fillna(0) * 100
        shares.plot.barh(stacked=True, ax=axis, legend=False, width=0.7,
                         color=[plt.cm.tab10(i % 10) if c != "unknown/other" else "#9ca3af" for i, c in enumerate(categories)])
        axis.set_xlabel(f"{title} within zone (%)"); axis.set_ylabel("Zone"); axis.set_title(title); axis.set_xlim(0, 100)
    handles, names = ax[0].get_legend_handles_labels()
    fig.legend(handles, names, title="Aircraft category", loc="center left", bbox_to_anchor=(1.0, 0.5))
    fig.suptitle("Aircraft category by zone (incl. unknown/other)")
    fig.tight_layout(); fig.savefig(OUT / "figures/category_share_by_zone.png", dpi=160, bbox_inches="tight"); plt.show()
else:
    print("View observations_enriched absent. Rebuild with src/prepare.py.")


### Interpretation (group): aircraft category by zone

*Placeholder.* How large is `unknown/other` per zone, and does it differ between zones (metadata coverage bias)? Do observation shares and aircraft shares tell the same story, or do a few frequently observed aircraft dominate? Which categories are too small to compare?

In [ ]:
# Block 5: wind arrows of the model grid for one hour and pressure level, over the observations of that hour.
import folium
from branca.colormap import LinearColormap
from branca.element import Element
MAP_LEVEL_HPA = 250     # ~10 km, typical cruise level; any level in config.json "weather"
MAP_HOUR_UTC = None     # e.g. "2026-10-22 13:00:00+00:00"; None = hour with the most matched observations
ARROW_DEG_PER_MS = 0.006  # arrow length: 50 m/s -> 0.3° latitude (~33 km)


def wind_arrow(lat, lon, speed_ms, from_deg):
    """Arrow pointing where the wind blows TO (from_deg + 180°), length proportional to speed."""
    to_rad = np.radians((from_deg + 180) % 360)
    length = speed_ms * ARROW_DEG_PER_MS
    scale_lon = 1 / np.cos(np.radians(lat))
    tip = (lat + length * np.cos(to_rad), lon + length * np.sin(to_rad) * scale_lon)
    head = []
    for side in (150, -150):  # two short barbs at the tip
        angle = to_rad + np.radians(side)
        head.append((tip[0] + 0.3 * length * np.cos(angle), tip[1] + 0.3 * length * np.sin(angle) * scale_lon))
    return [[(lat, lon), tip], [head[0], tip, head[1]]]


if HAS_WIND:
    hour = MAP_HOUR_UTC or wind_obs.wind_time_utc.value_counts().idxmax()
    with closing(sqlite3.connect(DB)) as db:
        grid = pd.read_sql_query("""SELECT grid_lat, grid_lon, height_m, wind_speed_ms, wind_dir_deg
            FROM weather_levels WHERE time_utc = ? AND level_hpa = ?""", db, params=(hour, MAP_LEVEL_HPA))
    hour_obs = wind_obs[wind_obs.wind_time_utc == hour].dropna(subset=["tailwind_ms"])
    if grid.empty:
        print(f"No grid wind for {hour} at {MAP_LEVEL_HPA} hPa; choose another MAP_HOUR_UTC or MAP_LEVEL_HPA.")
    else:
        print(f"Hour {hour}, {MAP_LEVEL_HPA} hPa: {len(grid)} grid points, {len(hour_obs)} observations")
        m = folium.Map(location=[46.8, 8.2], zoom_start=7)
        for airport in airports.itertuples():
            folium.Marker([airport.latitude_deg, airport.longitude_deg], tooltip=airport.ident).add_to(m)
        for point in grid.itertuples():
            tooltip = (f"{MAP_LEVEL_HPA} hPa (~{point.height_m:,.0f} m): {point.wind_speed_ms:.1f} m/s, "
                       f"from {point.wind_dir_deg:.0f}°, blowing toward {(point.wind_dir_deg + 180) % 360:.0f}°")
            for segment in wind_arrow(point.grid_lat, point.grid_lon, point.wind_speed_ms, point.wind_dir_deg):
                folium.PolyLine(segment, color="#111827", weight=2, tooltip=tooltip).add_to(m)
        limit = max(5.0, float(hour_obs.tailwind_ms.abs().max())) if len(hour_obs) else 5.0
        tail_colors = LinearColormap(["#dc2626", "#f3f4f6", "#2563eb"], vmin=-limit, vmax=limit,
                                     caption="Tailwind along track at the aircraft's own altitude (m/s; − = headwind)")
        for row in hour_obs.itertuples():
            folium.CircleMarker([row.latitude, row.longitude], radius=4, color=tail_colors(row.tailwind_ms),
                fill=True, fill_opacity=0.8, tooltip=(f"{row.zone}; {row.geo_altitude:,.0f} m; ground speed "
                f"{row.velocity:.0f} m/s; tailwind {row.tailwind_ms:.1f} m/s")).add_to(m)
        tail_colors.add_to(m)
        m.get_root().html.add_child(Element(
            '<div style="position:fixed;bottom:30px;left:30px;background:white;padding:12px;z-index:9999;max-width:320px">'
            f'<b>Modelled wind, {hour.replace("+00:00", "")} UTC, {MAP_LEVEL_HPA} hPa (~{grid.height_m.mean():,.0f} m)</b><br>'
            'Arrows point where the wind blows <b>to</b> (wind_dir_deg + 180°); wind_dir_deg is where it comes from.<br>'
            f'Arrow length proportional to speed (50 m/s ≈ {50 * ARROW_DEG_PER_MS:.1f}° latitude).<br>'
            'Circles: observations matched to this hour, coloured by tailwind at their own altitude.</div>'))
        m.save(str(OUT / "figures/wind_map.html"))
        display(m)


### Interpretation (group): wind map

*Placeholder.* Which hour and level are shown, and why? Does the arrow field match the direction pattern in block 3? The arrows show one pressure level, while each circle is coloured by the wind interpolated at its own altitude, so low aircraft can differ from the arrows. Model grid is 0.5°; do not read local detail into it.